# 03A 资料如何变成可检索的证据

本课从一个具体问题开始：**“学习记录保存在什么文件？”** 第二节的 Agent 可以逐个读文件；文件变多后，我们希望先找到最相关的几段。

这一份 Notebook 解决两个问题：如何保留原文位置，如何用可解释的数字给段落排序。只用 Python 标准库，不需要 API 或模型下载。

学习方式：先预测结果，再执行代码，再改变一个条件进行比较。这里的三个文件由课程自编，不是对任何真实产品的说明。


## 学习路线与配套课件

实践 3A：切块、关键词与相似度（`L03-P01`）。

建议先完成本节概念正课，再进入本实践小节。这个 Notebook 可用独立新内核从头运行。先预测，再执行代码、修改一个条件并解释结果。

对应课件稳定编号：L03.03-S01、L03.03-S02。页数调整不改变这些编号。

按“问题—代码—观察—练习—可复用结果”的顺序学习。先完成练习，再展开参考分析。回放、保存的真实记录和新的在线请求都会明确标注；在线请求默认关闭。

In [1]:
from pathlib import Path
import sys, json, math, hashlib
from collections import Counter
from dataclasses import dataclass
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "00-资料库使用说明.md").exists())
CHAPTER = ROOT / "03 资料检索与有依据的回答"
sys.path.insert(0, str(CHAPTER / "代码"))
from retrieval_lab import *
print("本课资料:", CHAPTER.relative_to(ROOT) / "data")


本课资料: chapters\03-retrieval\data


## 1. 模型没看到资料时，依据从哪里来

“请认真回答”不会自动把本地文件送给模型。我们先读取自己的资料，再让检索程序选择候选片段，最后把片段交给生成模型。

这是一条常见的 RAG（Retrieval-Augmented Generation，检索增强生成）流程。原始 RAG 论文研究了检索器与生成器结合的训练方法；本课搭建的是**应用层的检索后生成流程**，没有复现论文的联合训练，也不更新 DeepSeek 的参数。

先预测：文件里有答案，是否就保证检索能找到？下面先用完整句子的字面匹配试一试。


In [2]:
question = "学习记录保存在什么文件？"
text = (CHAPTER / "数据/study-guide.md").read_text(encoding='utf-8')
print("完整问句是否出现:", question in text)
print("关键词是否出现:", "学习记录" in text)
print(text.splitlines()[1])
# 问题和答案通常不是同一句话，因此全文完全匹配过于严格。


完整问句是否出现: False
关键词是否出现: True
学习记录保存在 progress.json；每条记录包含学习目标、最近一次练习和下一步计划。


## 2. Chunk 是有边界的一段资料

chunk 就是一次检索返回的文本片段。我们还要保存文件名、起止行号和内容版本。否则看见一句答案，也不知道它来自哪里。

先用按行切块理解边界：每块 3 行，相邻块重叠 1 行。步长等于 `3 - 1 = 2`。重叠是重复保留边界附近的信息，不能自动恢复所有跨段关系。真实 PDF、Markdown 和代码往往还需要按页、标题或函数边界切块。

下面直接展示课程使用的完整切块函数。`sha256` 是内容指纹，用于发现版本变化；它不证明内容真实。


In [3]:
@dataclass(frozen=True)
class Chunk:
    source: str
    start_line: int
    end_line: int
    text: str
    source_sha256: str

    @property
    def citation(self):
        return f"{self.source}:{self.start_line}-{self.end_line}"

    @property
    def id(self):
        # 内容变化时 ID 也变化，避免拿旧行号指向新版本。
        raw = f"{self.source_sha256}:{self.citation}"
        return hashlib.sha256(raw.encode()).hexdigest()[:16]

def chunk_text(text, source, lines_per_chunk=3, overlap=1):
    """按行演示切块；生产资料通常还需遵循章节、段落或语法边界。"""
    if not isinstance(lines_per_chunk, int) or not isinstance(overlap, int):
        raise TypeError("块大小与重叠必须为整数")
    if not 0 <= overlap < lines_per_chunk:
        raise ValueError("必须满足 0 <= overlap < lines_per_chunk")
    lines = text.splitlines()
    digest = hashlib.sha256(text.encode()).hexdigest()
    chunks = []
    step = lines_per_chunk - overlap
    for start in range(0, len(lines), step):
        end = min(start + lines_per_chunk, len(lines))
        content = "\n".join(lines[start:end])
        if content.strip():
            chunks.append(Chunk(source, start + 1, end, content, digest))
        if end == len(lines):
            break
    return chunks


In [4]:
tiny = "第一行：学习目标\n第二行：读取配置\n第三行：检查结果\n第四行：做小练习\n第五行：保存进度"
blocks = chunk_text(tiny, "tiny.txt", lines_per_chunk=3, overlap=1)
for block in blocks:
    print(block.citation, "ID:", block.id)
    print(block.text, "\n")
    # 切块保存的正文必须能从原文中按行取回。
    assert block.text == "\n".join(tiny.splitlines()[block.start_line-1:block.end_line])


tiny.txt:1-3 ID: 8fce99c907ebf469
第一行：学习目标
第二行：读取配置
第三行：检查结果 

tiny.txt:3-5 ID: e090bc38d1648ad0
第三行：检查结果
第四行：做小练习
第五行：保存进度 



### 练习：只改变块大小

先把 `lines_per_chunk` 改为 2、`overlap` 保持 1，再改为 5。记录块数，并说明：块更小会不会切断“如果……那么……”？块更大会不会带进无关内容？不要用“越大越好”总结。

随后尝试令重叠等于块大小。为什么必须报错？


In [5]:
student_size = 2  # TODO：再改成 5；先预测块数。
student_blocks = chunk_text(tiny, "tiny.txt", student_size, 1)
print([(b.start_line, b.end_line) for b in student_blocks])


[(1, 2), (2, 3), (3, 4), (4, 5)]


In [6]:
for size in [2, 5]:
    print(size, "行/块:", len(chunk_text(tiny, "tiny.txt", size, 1)), "块")
try:
    chunk_text(tiny, "tiny.txt", 3, 3)
except ValueError as error:
    print("重叠等于块大小，步长会变成 0:", error)
# 2 行产生 4 块，5 行产生 1 块。大小要结合问题与资料边界选择。


2 行/块: 4 块
5 行/块: 1 块
重叠等于块大小，步长会变成 0: 必须满足 0 <= overlap < lines_per_chunk


## 3. “词”必须先有一个明确约定

检索先把文本变成可比较的单元。本课中文使用连续两个汉字的片段，例如“学习记录”得到“学习、习记、记录”；英文标识符完整保留。这种做法容易解释，但“习记”并不是自然语言中的词。

**本课检索单元与第一节的语言模型 Token 不是一回事。** 我们没有调用 DeepSeek 的分词器，也没有计算 API 的输入 Token 数。观察 `_` 与中文标点如何影响输出。


In [7]:
def terms(text):
    """中文连续二字片段 + 英文/数字标识符。检索单元不等于 LLM Token。"""
    output = []
    for item in re.findall(r'[\u4e00-\u9fff]+|[a-zA-Z0-9_]+', text.lower()):
        if '\u4e00' <= item[0] <= '\u9fff':
            output.extend(item[i:i+2] for i in range(len(item)-1))
            if len(item) == 1:
                output.append(item)
        else:
            output.append(item)
    return output


In [8]:
for sample in ["学习记录", "load_settings 读取配置。", "学习，记录", "进度"]:
    print(sample, "->", terms(sample))


学习记录 -> ['学习', '习记', '记录']
load_settings 读取配置。 -> ['load_settings', '读取', '取配', '配置']
学习，记录 -> ['学习', '记录']
进度 -> ['进度']


## 4. 从词频到 TF-IDF

用一个字典存一段文本的向量：键是检索单元，值是权重。没有出现的单元权重为 0。

TF 是该单元在这段文本中的出现次数。DF 是包含它的片段数量。若一个词几乎每段都有，它区分段落的能力通常较弱；若它只在少数段出现，它更能区分这些段。

本课使用平滑公式：

$$\mathrm{idf}(t)=\ln\frac{1+N}{1+\mathrm{df}(t)}+1,\qquad w_{t,d}=\mathrm{tf}(t,d)\,\mathrm{idf}(t)$$

N 为片段总数，ln 为自然对数。平滑版本与 scikit-learn 默认 IDF 约定一致；不要把它与所有教材中的 TF-IDF 变体混为同一个公式。

我们先用两个英文短句手算。猜一猜：“cat”与“dog”哪个 IDF 更高？


In [9]:
documents = ["cat cat dog", "dog bird"]
counts = [Counter(terms(d)) for d in documents]
df = Counter(t for count in counts for t in count)
idf = {t: math.log((1 + len(documents)) / (1 + n)) + 1 for t,n in df.items()}
weights = [{t: n * idf[t] for t,n in count.items()} for count in counts]
print("词频:", counts)
print("文档频率:", df)
print("IDF:", idf)
print("加权结果:", weights)
assert idf["cat"] > idf["dog"]


词频: [Counter({'cat': 2, 'dog': 1}), Counter({'dog': 1, 'bird': 1})]
文档频率: Counter({'dog': 2, 'cat': 1, 'bird': 1})
IDF: {'cat': 1.4054651081081644, 'dog': 1.0, 'bird': 1.4054651081081644}
加权结果: [{'cat': 2.8109302162163288, 'dog': 1.0}, {'dog': 1.0, 'bird': 1.4054651081081644}]


## 5. 余弦相似度比较方向

直接点乘会受向量长度影响。余弦先除以两个向量的长度，再比较方向：

$$\cos(q,d)=\frac{q\cdot d}{\lVert q\rVert_2\lVert d\rVert_2}$$

先把两个向量分别除以自身长度，得到单位向量，之后点乘就是余弦。本课所有 TF-IDF 权重非负，所以其余弦在 0 到 1 之间；一般实向量的余弦可以为负。

相似度不是“答案正确的概率”。分数 0.8 不表示有 80% 的概率答对。零向量无法用普通公式相除；程序把没有可比较信息的情况记作 0。


In [10]:
def normalize(vector):
    length = math.sqrt(sum(v*v for v in vector.values()))
    return {term: value/length for term, value in vector.items()} if length else {}

def cosine(left, right):
    a, b = normalize(left), normalize(right)
    return sum(value * b.get(term, 0) for term, value in a.items())


In [11]:
a = {"cat": 1.0, "dog": 2.0}
b = {"cat": 2.0, "dog": 4.0}
print("同方向、不同长度:", cosine(a, b))
print("没有共同坐标:", cosine({"cat":1}, {"bird":1}))
print("零向量:", cosine({}, a))
assert math.isclose(cosine(a,b), 1.0)


同方向、不同长度: 0.9999999999999999
没有共同坐标: 0.0
零向量: 0


### 把 TF-IDF 权重接到最终相似度

配套课件：L03.03-S01、L03.03-S02。

继续使用上面的两份文本 cat cat dog / dog bird。TF-IDF 只是得到向量，排序还需要度量。现在把查询设为 cat，使用相同的 IDF 坐标计算余弦，逐项对照公式 $\cos(q,d)=q\cdot d/(\|q\|\|d\|)$。

In [12]:
case_query = {"cat": idf["cat"]}
for case_number, case_document in enumerate(weights, 1):
    case_dot = sum(case_query.get(t, 0) * w for t, w in case_document.items())
    case_qnorm = math.sqrt(sum(w*w for w in case_query.values()))
    case_dnorm = math.sqrt(sum(w*w for w in case_document.values()))
    case_score = case_dot / (case_qnorm * case_dnorm)
    print("文档", case_number, "点积:", round(case_dot, 4),
          "两侧长度:", round(case_qnorm, 4), round(case_dnorm, 4),
          "余弦:", round(case_score, 4))
    assert math.isclose(case_score, cosine(case_query, case_document))
assert cosine(case_query, weights[0]) > cosine(case_query, weights[1])


文档 1 点积: 3.9507 两侧长度: 1.4055 2.9835 余弦: 0.9422
文档 2 点积: 0.0 两侧长度: 1.4055 1.7249 余弦: 0.0


**结果解读**

第一份文本与 cat 查询相似度约 0.9422，第二份为 0。出现两次 cat 增加对应坐标的权重，但不是“有 94% 概率答对”。检索分数不能代替答案支持度。

**小练习**

查询改为 dog。先看两份文本的 dog 权重都为 1，再解释为什么余弦仍可能不同。

<details><summary>完成后展开参考分析</summary>

分子相同不代表余弦相同；两份文档的其他坐标不同，向量长度也不同。

</details>

公式与实现来源见 [配套素材来源](../../资料来源.md)。

## 6. 建索引与查问题是两步

建索引时统计资料的词表、IDF 和向量；查询时使用**同一份**词表和 IDF 编码问题，才能逐个坐标比较。不能为问题重新建立一套坐标后直接点乘。

下面打印完整索引实现。小资料直接遍历全部片段就能讲清算法；更多数据需要考虑索引结构和更新成本，但本课不要求安装向量数据库。


In [13]:
class TfidfIndex:
    """TF 用原始词频，IDF = ln((1+N)/(1+df))+1；与 sklearn 默认平滑 IDF 一致。"""
    def __init__(self, chunks):
        self.chunks = list(chunks)
        counts = [Counter(terms(c.text)) for c in self.chunks]
        frequency = Counter(term for count in counts for term in count)
        self.idf = {term: math.log((1+len(counts))/(1+df))+1 for term, df in frequency.items()}
        self.vectors = [self.encode_counts(count) for count in counts]

    def encode_counts(self, count):
        return normalize({term: n*self.idf[term] for term, n in count.items() if term in self.idf})

    def search(self, query, k=3):
        if not isinstance(k, int) or k < 1:
            raise ValueError("k 必须为正整数")
        vector = self.encode_counts(Counter(terms(query)))
        hits = []
        for chunk, candidate in zip(self.chunks, self.vectors):
            score = sum(v*candidate.get(term, 0) for term, v in vector.items())
            if score > 0:
                hits.append({'chunk': chunk, 'score': score})
        return sorted(hits, key=lambda h: (-h['score'], h['chunk'].citation))[:k]


In [14]:
chunks = load_corpus(CHAPTER / "数据")
index = TfidfIndex(chunks)
hits = index.search(question, k=3)
for rank, hit in enumerate(hits, 1):
    print(rank, round(hit["score"], 4), hit["chunk"].citation)
    print(hit["chunk"].text, "\n")
print("无词表交集:", index.search("quasar_z9"))


1 0.3332 study-guide.md:1-3
# StudyBox 使用说明（本课程自编资料）
学习记录保存在 progress.json；每条记录包含学习目标、最近一次练习和下一步计划。
没有完成的练习不会标记为已掌握；学生可以保留尝试记录后继续。 

2 0.1734 unrelated.md:3-4
图书借阅期限为三十天，延期请联系图书管理员。
本文件没有介绍 StudyBox 的配置和学习记录。 

3 0.0953 study-guide.md:7-9
没有设置环境变量时使用相对路径 settings.json；相对路径以进程当前工作目录为起点。
文件不存在时返回默认配置，其中 timeout 为 30；已有文件中的 timeout 可以覆盖这一取值。
 

无词表交集: []


## 7. 实用检查：生成可回查的切块清单

真实检索系统不能只保存向量。至少还要知道每个片段来自哪份资料、覆盖哪些行、对应哪一版原文。下面把当前语料整理成一份最小切块清单，并检查片段 ID 是否唯一。

这份清单用于排错和更新索引，不判断资料是否真实，也不代替访问权限控制。


In [15]:
chunk_manifest = [
    {
        "chunk_id": chunk.id,
        "citation": chunk.citation,
        "source_sha256": chunk.source_sha256[:12],
        "characters": len(chunk.text),
        "preview": chunk.text.splitlines()[0][:28],
    }
    for chunk in chunks
]
for row in chunk_manifest:
    print(row)
assert len({row["chunk_id"] for row in chunk_manifest}) == len(chunk_manifest)
assert all(row["citation"] and row["source_sha256"] for row in chunk_manifest)


{'chunk_id': '7faae8fb08e14fb1', 'citation': 'faq.md:1-3', 'source_sha256': '3725b02d4b63', 'characters': 73, 'preview': '# 常见问题（本课程自编资料）'}
{'chunk_id': 'ac008938b72ac74b', 'citation': 'faq.md:3-5', 'source_sha256': '3725b02d4b63', 'characters': 58, 'preview': '答：演示客户端使用 timeout 参数设置等待上限；超'}
{'chunk_id': 'e199e08ebbcdc315', 'citation': 'faq.md:5-7', 'source_sha256': '3725b02d4b63', 'characters': 53, 'preview': '问：如何继续昨天的学习？'}
{'chunk_id': '1d3b1abd0505321d', 'citation': 'faq.md:7-9', 'source_sha256': '3725b02d4b63', 'characters': 51, 'preview': ''}
{'chunk_id': '020ea4b9b647b978', 'citation': 'study-guide.md:1-3', 'source_sha256': 'ec2bfa2c6e0e', 'characters': 102, 'preview': '# StudyBox 使用说明（本课程自编资料）'}
{'chunk_id': '589458d9685495ec', 'citation': 'study-guide.md:3-5', 'source_sha256': 'ec2bfa2c6e0e', 'characters': 38, 'preview': '没有完成的练习不会标记为已掌握；学生可以保留尝试记录后继'}
{'chunk_id': '5e71da2764745d36', 'citation': 'study-guide.md:5-7', 'source_sha256': 'ec2bfa2c6e0e', 'characters': 98, 'preview

**如何使用这份结果**

- 资料更新后，先比较 `source_sha256`，再决定是否重建对应片段。
- 回答出现问题时，用 `citation` 回到原文，而不是只看相似度分数。
- `preview` 只帮助快速浏览，真正核验仍要读取完整片段。


## 本次小结

现在可以从问题找到若干片段，并把片段定位回原文。你已经亲手检查了切块、词频、平滑 IDF、向量归一化和排序。

仍有一个缺口：问题换成“昨天没学完的内容今天怎么接着学”，可能不再包含“学习记录”这些词。下一份 Notebook 用真实句向量比较这个现象，不能提前假设语义检索一定更好。

参考：[RAG 原始论文](https://arxiv.org/abs/2005.11401)、[Stanford 信息检索教材](https://nlp.stanford.edu/IR-book/html/htmledition/tf-idf-weighting-1.html)、[scikit-learn TF-IDF 约定](https://scikit-learn.org/stable/modules/feature_extraction.html#tfidf-term-weighting)。核验日期：2026-09-07。
